In [ ]:
import pandas as pd
import os
import shutil
import smtplib
import win32com.client
from openpyxl import load_workbook
from email.message import EmailMessage
from datetime import datetime
import time


def create_directories():
    os.makedirs("output/excel", exist_ok=True)
    os.makedirs("output/pdf", exist_ok=True)
    os.makedirs("logs", exist_ok=True)


def start_excel():
    excel = win32com.client.DispatchEx("Excel.Application")
    excel.Visible = False
    excel.DisplayAlerts = False
    return excel


def create_log(log_file):
    if not os.path.exists(log_file):
        with open(log_file, "w") as f:
            f.write("timestamp,employee_id,name,email,status\n")


def generate_payslip(row, template, excel_folder):
    employee_name = row["Personnel Name"]

    excel_file = os.path.abspath(f"{excel_folder}/Payslip_{employee_name}.xlsx")

    shutil.copy(template, excel_file)

    wb = load_workbook(excel_file)
    ws = wb.active

    ws['B3'] = row['Personnel Name']
    ws['B4'] = row['Employee Id']
    ws['B5'] = row['Job Title']
    
    ws['B7'] = row['Total Days of the Month']
    ws['B8'] = row['Days Worked']
    ws['B9'] = row['Total Days of the Month'] - row['Days Worked']
    ws['B11'] = row['New Base Fees']
    
    ws['B16'] = row['Service Fees (a1)']
    ws['B17'] = row['Overtime/Weekend Work (a3)']
    ws['B18'] = row['Public Holiday']
    ws['B19'] = row['OTHER ALLOWANCE']
    ws['B20'] = row['Accommodation Allowance']
    ws['B21'] = row['Transport (a2)']
    ws['B22'] = row['Omitted overtime']
    ws['B23'] = row['Omitted reimbursable']
    ws['B24'] = row['Special Management Bonus']
    ws['B26'] = row['Gross Service Fees']
    
    ws['E16'] = row['5% WHT']
    ws['E17'] = row['Deduction']
    ws['E26'] = row['Net Pay']

    wb.save(excel_file)

    return excel_file


def convert_to_pdf(excel, excel_file, pdf_folder, employee_name):

    pdf_file = os.path.abspath(f"{pdf_folder}/Payslip_{employee_name}.pdf")

    workbook = excel.Workbooks.Open(excel_file)
    workbook.ExportAsFixedFormat(0, pdf_file)
    workbook.Close(False)

    return pdf_file


def send_email(row, pdf_file, sender_email, password, cc_emails):

    msg = EmailMessage()

    emails = [e.strip() for e in str(row["email"]).split(";")]

    msg["Subject"] = "Monthly Payslip"
    msg["From"] = sender_email
    msg["To"] = ",".join(emails)
    msg["Cc"] = ",".join(cc_emails)

    msg.set_content(f"""
Hello {employee_name},

Please find attached your payslip.

Regards
Finance Department
""")

    with open(pdf_file, "rb") as f:
        msg.add_attachment(
            f.read(),
            maintype="application",
            subtype="pdf",
            filename=os.path.basename(pdf_file)
        )

    smtp = smtplib.SMTP_SSL("smtp.gmail.com", 465)
    smtp.login(sender_email, password)
    smtp.send_message(msg)
    smtp.quit()


def log_email(log_file, employee_id, name, email, status):

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    with open(log_file, "a") as f:
        f.write(f"{timestamp},{employee_id},{name},{email},{status}\n")


def process_payroll():

    template = "templates/payslip_template.xlsx"
    excel_folder = "output/excel"
    pdf_folder = "output/pdf"
    log_file = "logs/email_log.csv"

    sender_email = "yourcompany@gmail.com"
    password = "your_app_password"

    cc_emails = ["hr@company.com"]

    create_directories()

    create_log(log_file)

    df = pd.read_csv("data/payroll.csv")

    excel = start_excel()

    for _, row in df.iterrows():

        try:

            excel_file = generate_payslip(row, template, excel_folder)

            pdf_file = convert_to_pdf(excel, excel_file, pdf_folder, row["employee_name"])

            send_email(row, pdf_file, sender_email, password, cc_emails)

            status = "SENT"

        except Exception as e:

            status = f"FAILED - {str(e)}"

        log_email(log_file, row["employee_id"], row["name"], row["email"], status)

        time.sleep(3)

    excel.Quit()


if __name__ == "__main__":
    process_payroll()